# Fine-tuning a pre-trained CNN for MNIST and saving a `.pth` model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FernandaChacara/PML/blob/main/mnist-cnn-assignment/train_mnist_resnet18.ipynb)

This notebook implements the training part of the assignment. The objective is to fine-tune a pre-trained convolutional neural network on the MNIST handwritten digit dataset and save the trained model weights as a `.pth` file. This saved file will later be loaded by a Gradio application deployed on Hugging Face Spaces.



## 1. Import libraries and set up the environment

This first section imports the libraries required for the full pipeline. PyTorch is used to build, fine-tune and save the CNN model. Torchvision is used to download MNIST and load the pre-trained ResNet18 architecture. The random seed is fixed so that the results are more reproducible when the notebook is re-run.


In [ ]:

# !pip install torch torchvision matplotlib tqdm

# General utilities
import os
import random
import numpy as np

# PyTorch core libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

# Torchvision provides datasets, image transforms and pre-trained models
from torchvision import datasets, transforms, models

# tqdm is used only to display progress bars during training and evaluation
from tqdm.auto import tqdm

# Matplotlib is used to visualize a few example images from MNIST
import matplotlib.pyplot as plt

# A fixed seed helps make the experiment more reproducible.
# The exact result can still vary slightly depending on GPU/backend behavior.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Use GPU if Colab provides one; otherwise, use CPU.
# The same notebook works in both situations, but GPU training is faster.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Load and preprocess the MNIST dataset

MNIST contains 28×28 grayscale images of handwritten digits from 0 to 9. However, the pre-trained ResNet18 model was originally trained on ImageNet, where images are much larger. For this reason, each MNIST image is resized to 224×224 pixels before being passed to the model.

The images remain grayscale with one input channel. Later, the first convolutional layer of ResNet18 will be adapted from three input channels to one input channel.


In [ ]:
# These transformations are applied to every MNIST image before it enters the model.
# 1. Resize: ResNet18 expects larger inputs than the original 28x28 MNIST images.
# 2. ToTensor: converts the PIL image into a PyTorch tensor.
# 3. Normalize: standard MNIST mean and standard deviation are used to scale pixel values.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307], std=[0.3081]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1307], std=[0.3081]),
])

# Download MNIST. The training set contains 60,000 images and the test set contains 10,000 images.
data_dir = "./data"
full_train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=train_transform)
test_dataset = datasets.MNIST(root=data_dir, train=False, download=True, transform=test_transform)

# The original training set is split into training and validation subsets.
# The validation subset is used to monitor performance during training.
train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# DataLoaders create mini-batches of images.
# shuffle=True is used for the training loader so the model sees the data in a different order each epoch.
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Test images: {len(test_dataset)}")


## 3. Visualize a few training examples

This step is not strictly necessary for training, but it is useful to confirm that the dataset has been loaded correctly and that the labels correspond to handwritten digits.


In [ ]:
# Display a few examples from the MNIST training set.
# The images shown here are normalized tensors, so we undo the normalization only for visualization.
examples, labels = next(iter(train_loader))

plt.figure(figsize=(10, 4))
for i in range(8):
    img = examples[i].squeeze().numpy()
    img = img * 0.3081 + 0.1307  # undo MNIST normalization for display
    plt.subplot(2, 4, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"Label: {labels[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 4. Build and adapt the pre-trained CNN

The selected pre-trained CNN is **ResNet18**. It was originally trained on ImageNet, a large image classification dataset. This assignment asks for fine-tuning, so we start from that pre-trained model instead of training a CNN from zero.

Two architectural changes are required:

1. **Input layer adaptation:** ImageNet images are RGB with three channels, while MNIST images are grayscale with one channel. The first convolutional layer is changed from 3 input channels to 1 input channel.
2. **Output layer adaptation:** ImageNet has 1000 classes, while MNIST has only 10 classes. The final fully connected layer is replaced by a new layer with 10 outputs, one for each digit.


In [ ]:
def build_model():
    """Build a ResNet18 model adapted for MNIST digit classification.

    The function starts from ImageNet pre-trained ResNet18 weights, adapts the
    first convolutional layer to accept grayscale images, and replaces the final
    classification layer with a 10-class output layer for digits 0-9.
    """

    # Load ImageNet pre-trained ResNet18.
    # These pre-trained weights provide useful low-level visual filters.
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)

    # ResNet18 normally expects 3-channel RGB images.
    # MNIST images are grayscale, so we replace the first convolutional layer.
    old_conv = model.conv1
    new_conv = nn.Conv2d(
        in_channels=1,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )

    # To initialize the new grayscale convolution, we average the original RGB filters.
    # This preserves information from the pre-trained filters instead of starting randomly.
    with torch.no_grad():
        new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))

    model.conv1 = new_conv

    # Replace the original ImageNet classifier with a classifier for 10 MNIST classes.
    model.fc = nn.Linear(model.fc.in_features, 10)

    return model

# Build the model and move it to the selected device.
model = build_model().to(device)

# CrossEntropyLoss is appropriate for multi-class classification.
criterion = nn.CrossEntropyLoss()

# Adam is used as the optimizer for fine-tuning all model parameters.
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print("Final classification layer:")
print(model.fc)


## 5. Define training and evaluation functions

The training function performs the standard deep learning training loop: forward pass, loss calculation, backpropagation and optimizer update. The evaluation function measures loss and accuracy without updating the model weights.


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train the model for one epoch and return average loss and accuracy."""

    # model.train() activates training behavior, such as gradient tracking and dropout/batch norm updates.
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        # Move the batch to GPU or CPU depending on the available device.
        images, labels = images.to(device), labels.to(device)

        # Reset gradients from the previous batch.
        optimizer.zero_grad()

        # Forward pass: the model produces raw class scores, also called logits.
        outputs = model(images)

        # Compute the difference between predicted scores and true labels.
        loss = criterion(outputs, labels)

        # Backpropagation: compute gradients for all trainable parameters.
        loss.backward()

        # Update the model parameters using the calculated gradients.
        optimizer.step()

        # Accumulate loss and accuracy statistics for reporting.
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    """Evaluate the model and return average loss and accuracy."""

    # model.eval() switches the model to evaluation mode.
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # torch.no_grad() disables gradient calculation, making evaluation faster and lighter.
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


## 6. Fine-tune the model for a few epochs

The assignment asks for the CNN to be fine-tuned for “a couple of epochs”. Here, the model is trained for two epochs. This is enough to demonstrate the complete pipeline: data loading, transfer learning, training, evaluation and model saving.

If more accuracy is needed, the number of epochs can be increased, but that will make the notebook take longer to run.


In [ ]:
# The assignment asks for a couple of epochs, so two epochs are used here.
num_epochs = 2

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")

    # Train the model on the training subset.
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # Evaluate the model on the validation subset.
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    # Store the metrics so they can be inspected later if needed.
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train loss: {train_loss:.4f} | Train accuracy: {train_acc:.4f}")
    print(f"Val loss:   {val_loss:.4f} | Val accuracy:   {val_acc:.4f}")
    print("-" * 60)


## 7. Test the final model

After training, the model is evaluated on the separate MNIST test set. The test set is not used for training or validation, so it provides a final estimate of the model's performance on unseen data.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Final test loss: {test_loss:.4f}")
print(f"Final test accuracy: {test_acc:.4f}")


## 8. Save the trained model as a `.pth` file

This is the key step for deployment. The trained weights are saved using `torch.save`. The saved checkpoint also includes useful metadata, such as the model architecture, the number of classes, the image size and the normalization values.

The Hugging Face Space will later load this `.pth` file and use it to make predictions in the Gradio interface.


In [ ]:
# Name of the file that will be uploaded to the Hugging Face Space.
model_path = "mnist_resnet18.pth"

# Instead of saving only the raw weights, this checkpoint saves the weights plus metadata.
# The metadata helps the deployment app know how the model was trained and how inputs should be processed.
checkpoint = {
    "model_state_dict": model.state_dict(),
    "architecture": "resnet18_grayscale_mnist",
    "num_classes": 10,
    "normalization_mean": [0.1307],
    "normalization_std": [0.3081],
    "input_size": [224, 224],
    "epochs": num_epochs,
    "test_accuracy": test_acc,
}

# Save the checkpoint as a .pth file.
torch.save(checkpoint, model_path)

print(f"Saved model to: {model_path}")
print(f"File size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")


## 9. Confirm that the saved model can be loaded again

This optional verification step checks that the `.pth` file was saved correctly. A new model with the same architecture is created, the saved weights are loaded, and the model is switched to evaluation mode.

This is the same logic used later in the Hugging Face Gradio app.


In [ ]:
# Load the checkpoint back from disk.
loaded_checkpoint = torch.load(model_path, map_location=device)

# Rebuild the same model architecture and load the saved weights.
loaded_model = build_model().to(device)
loaded_model.load_state_dict(loaded_checkpoint["model_state_dict"])
loaded_model.eval()

print("Saved model was loaded successfully.")
print("Checkpoint metadata:")
print({key: value for key, value in loaded_checkpoint.items() if key != "model_state_dict"})


## 10. Download the `.pth` file from Colab

After this notebook is executed in Google Colab, download `mnist_resnet18.pth`. This file must be uploaded to the Hugging Face Space together with the Gradio application files.


In [ ]:
# This cell works only in Google Colab.
# It opens a browser download dialog for the trained .pth file.
try:
    from google.colab import files
    files.download(model_path)
except Exception as e:
    print("This download command only works in Google Colab.")
    print("If you are not in Colab, manually locate the file:", model_path)
    print(e)


## 11. Files used for deployment

The Hugging Face Space should contain these files:

```text
app.py
requirements.txt
mnist_resnet18.pth
examples/
```

The Gradio app will load `mnist_resnet18.pth`, preprocess the uploaded or drawn image using the same resize and normalization steps, and output the estimated probability for each digit from 0 to 9.
